# 02 — N-BEATS swap experiment (SARIMA + N-BEATS hybrid)

Swap pada baseline yang direproduksi di `01_baseline_bilstm.ipynb`: **BiLSTM → N-BEATS** sebagai pembelajaran residu pada hybrid SARIMA.

Protokol (identik dengan blok orizonturi di notebook 01, supaya kedua notebook bisa dibandingkan):
- data: `../data/transformed_data.csv` (skala Box-Cox, 1.826 baris harian), target **NO2**
- split: 720 hari terakhir = jendela uji; order SARIMA **(2,1,1)(0,0,0,12)** — sama seperti notebook 01
- evaluasi: **rolling-origin**, satu origin tiap 30 hari, h langkah ke depan untuk h ∈ {1, 7, 30}
- masukan koreksi: N-BEATS `input_size=90` (sama dengan panjang sekuens BiLSTM di notebook 01)
- metrik biaya: jumlah parameter + waktu latih

Baseline pembanding dibaca dari `../results/01_baseline_horizon_metrics.csv` (hasil notebook 01),
**bukan** angka yang dilaporkan Necula et al. (2025): blok orizonturi mereka tidak dapat direproduksi
(lihat `03_Catatan/TEMUAN_FASE1.md`).

In [ ]:
# Bootstrap: jalan di Kaggle / Colab / lokal.
# Kalau '../data' belum ada (runtime baru), clone repo dulu supaya path relatif '../data/...' ketemu.
import os, subprocess
if not os.path.isdir('../data'):
    REPO = 'https://github.com/madajabbar/hybrid-sarima-nbeats_Anc.git'
    ROOT = '/kaggle/working' if os.path.isdir('/kaggle/working') else '/content'
    DEST = os.path.join(ROOT, 'AnC')
    if not os.path.isdir(DEST):
        subprocess.run(['git', 'clone', '-q', '-b', 'main', REPO, DEST], check=True)
    os.chdir(os.path.join(DEST, 'notebooks'))
print('cwd:', os.getcwd(), '| data:', os.path.isfile('../data/transformed_data.csv'))

In [ ]:
import os, time, warnings
warnings.filterwarnings('ignore')
import numpy as np, pandas as pd
from scipy.stats import boxcox
from statsmodels.tsa.statespace.sarimax import SARIMAX
from neuralforecast import NeuralForecast
from neuralforecast.models import NBEATS
import torch

DATA = '../data/transformed_data.csv'
FEATURE = 'NO2'                 # sama dengan baseline yang direproduksi di notebook 01
TEST_SIZE_DAYS = 720            # jendela evaluasi (identik dengan blok orizonturi notebook 01)
ORIGIN_STEP = 30                # satu origin tiap 30 hari
HORIZONS = [1, 7, 30]
INPUT_SIZE = 90                 # = panjang sekuens BiLSTM di notebook 01
SARIMA_ORDER, SEASONAL_ORDER = (2, 1, 1), (0, 0, 0, 12)   # identik dengan notebook 01
ACC = 'gpu' if torch.cuda.is_available() else 'cpu'

raw = pd.read_csv(DATA)
raw.columns = [c.strip('\ufeff') for c in raw.columns]
raw['Date'] = pd.to_datetime(raw['Date'], format='%d-%m-%y')
raw = raw.sort_values('Date').set_index('Date')
transformed_data = pd.DataFrame(index=raw.index)
for c in raw.columns:
    pv = raw[c] + abs(raw[c].min()) + 1
    transformed_data[c], _ = boxcox(pv)

train_data = transformed_data.iloc[:-TEST_SIZE_DAYS]
test_data = transformed_data.iloc[-TEST_SIZE_DAYS:]
print(len(transformed_data), 'baris | train', len(train_data), '| test', len(test_data), '| accelerator:', ACC)

In [ ]:
def hmetrics(actual, pred):
    actual, pred = np.asarray(actual, float), np.asarray(pred, float)
    return (float(np.sqrt(np.mean((actual - pred) ** 2))),
            float(np.mean(np.abs(actual - pred))),
            float(np.mean(np.abs((actual - pred) / actual)) * 100))

In [ ]:
# Baseline: metrik hasil reproduksi notebook 01 (bukan angka Necula 2025 — lihat TEMUAN_FASE1.md)
HB = '../results/01_baseline_horizon_metrics.csv'
BM = '../results/01_baseline_metrics.csv'
baseline = pd.read_csv(HB) if os.path.exists(HB) else None
base_cost = pd.read_csv(BM) if os.path.exists(BM) else None
if baseline is None:
    print('PERINGATAN: hasil notebook 01 belum ada (' + HB + ').')
    print('Jalankan notebook 01 dulu, lalu notebook ini. Perbandingan akan dilewati.')
else:
    print('baseline notebook 01 (orizonturi):')
    print(baseline[baseline.target == FEATURE].to_string(index=False))

In [ ]:
# SARIMA + N-BEATS: evaluare rolling-origin, protokol identik dengan notebook 01
sarima_model = SARIMAX(train_data[FEATURE], order=SARIMA_ORDER, seasonal_order=SEASONAL_ORDER).fit(disp=False)

rows, cost_rows = [], []
for h in HORIZONS:
    # N-BEATS dilatih sekali per orizont, pada residu data train
    resid_train = (train_data[FEATURE] - sarima_model.fittedvalues).dropna()
    rd = pd.DataFrame({'unique_id': FEATURE, 'ds': resid_train.index, 'y': resid_train.values})
    stacks = ('identity',) if h == 1 else ('trend', 'seasonality')  # neuralforecast: h=1 tak menerima trend/seasonality
    model = NBEATS(h=h, input_size=INPUT_SIZE, max_steps=200, stack_types=stacks,
                   enable_progress_bar=False, logger=False, accelerator=ACC)
    nf = NeuralForecast(models=[model], freq='D')
    t0 = time.time(); nf.fit(rd); train_s = time.time() - t0
    n_params = int(sum(p.numel() for p in nf.models[0].parameters()))
    cost_rows.append({'horizon': h, 'model': 'NBEATS', 'n_params': n_params, 'train_s': round(train_s, 1)})
    print(f'h={h}: N-BEATS dilatih {train_s:.1f}s, {n_params:,} parameter')

    for origin in range(0, len(test_data) - h, ORIGIN_STEP):
        hist = pd.concat([train_data[FEATURE], test_data[FEATURE].iloc[:origin]])
        res = sarima_model.apply(hist)                        # parameter sama, histori diperpanjang
        sarima_h = float(res.forecast(steps=h).iloc[-1])
        resid_now = (hist - res.fittedvalues).dropna()
        corr = 0.0
        if len(resid_now) >= INPUT_SIZE:
            nd = pd.DataFrame({'unique_id': FEATURE, 'ds': resid_now.index[-INPUT_SIZE:],
                               'y': resid_now.values[-INPUT_SIZE:]})
            corr = float(nf.predict(df=nd)['NBEATS'].values[-1])   # koreksi residu untuk orizont h
        rows.append({'horizon': h,
                     'origin_date': str(test_data.index[origin].date()),
                     'target_date': str(test_data.index[origin + h - 1].date()),
                     'actual': float(test_data[FEATURE].iloc[origin + h - 1]),
                     'sarima': sarima_h, 'hybrid': sarima_h + corr, 'nbeats_correction': corr})

swap_pred = pd.DataFrame(rows)
swap_metrics = []
for h in HORIZONS:
    d = swap_pred[swap_pred.horizon == h]
    for model_name, col in [('SARIMA', 'sarima'), ('SARIMA+NBEATS', 'hybrid')]:
        rmse, mae, mape = hmetrics(d.actual, d[col])
        swap_metrics.append({'target': FEATURE, 'horizon': h, 'model': model_name,
                             'rmse': rmse, 'mae': mae, 'mape': mape, 'n_origins': len(d)})
        print(f'  h={h:>2} {model_name:14s} RMSE={rmse:.4f} MAE={mae:.4f} MAPE={mape:.2f}%  (n={len(d)})')

swap_metrics_df = pd.DataFrame(swap_metrics)
swap_metrics_df.to_csv('../results/02_nbeats_swap_metrics.csv', index=False)
swap_pred.to_csv('../results/02_nbeats_swap_predictions.csv', index=False)
pd.DataFrame(cost_rows).to_csv('../results/02_nbeats_swap_cost.csv', index=False)
print('tersimpan: ../results/02_nbeats_swap_metrics.csv + _predictions.csv + _cost.csv')

In [ ]:
# Perbandingan: SARIMA+N-BEATS vs baseline hasil reproduksi (SARIMA+BiLSTM) + biaya
if baseline is not None:
    rows = []
    for h in HORIZONS:
        b_sar = baseline[(baseline.horizon == h) & (baseline.model == 'SARIMA')]
        b_hyb = baseline[(baseline.horizon == h) & (baseline.model == 'SARIMA+BiLSTM')]
        o_sar = swap_metrics_df[(swap_metrics_df.horizon == h) & (swap_metrics_df.model == 'SARIMA')]
        o_hyb = swap_metrics_df[(swap_metrics_df.horizon == h) & (swap_metrics_df.model == 'SARIMA+NBEATS')]
        if b_sar.empty or b_hyb.empty or o_sar.empty or o_hyb.empty:
            continue
        rows.append({'horizon': h,
                     'sarima_rmse': float(o_sar.rmse.iloc[0]),
                     'bilstm_rmse': float(b_hyb.rmse.iloc[0]),
                     'nbeats_rmse': float(o_hyb.rmse.iloc[0]),
                     'delta_nbeats_vs_bilstm': float(o_hyb.rmse.iloc[0] - b_hyb.rmse.iloc[0]),
                     'mape_bilstm': float(b_hyb.mape.iloc[0]),
                     'mape_nbeats': float(o_hyb.mape.iloc[0])})
    comparison = pd.DataFrame(rows)
    print(comparison.to_string(index=False))
    comparison.to_csv('../results/02_nbeats_vs_baseline.csv', index=False)

    cost = pd.DataFrame(cost_rows)
    if base_cost is not None and (base_cost.model == 'SARIMA+BiLSTM').any():
        b = base_cost[base_cost.model == 'SARIMA+BiLSTM'].iloc[0]
        print(f'Biaya | BiLSTM: {int(b.n_params)} parameter, fit {b.fit_time_s}s | '
              f'N-BEATS: {cost.n_params.max()} parameter (h=30), latih {cost.train_s.sum():.1f}s untuk 3 orizonturi')
    else:
        print('Biaya N-BEATS:')
        print(cost.to_string(index=False))
else:
    print('Perbandingan dilewati: jalankan notebook 01 dulu.')

## Langkah berikutnya

1. Perluas ke PM10 / PM25 setelah baseline-nya direproduksi (protokol sama).
2. Tambah beberapa seed (`NBEATS(..., random_seed=...)`) → laporkan rata-rata ± simpangan.
3. Uji Diebold–Mariano pada pasangan galat per-origin (`02_nbeats_swap_predictions.csv` vs milik notebook 01).
4. Opsional: baris N-HiTS untuk perbandingan akurasi vs kompleksitas.